## Simulación de MonteCarlo

In [15]:
import numpy as np
from numpy.linalg import cholesky
import os

In [ ]:
S_0 = np.array([1,1,1])

In [16]:
def MonteCarlo(initial_maturity, S_0, num_sims, num_assets, num_asian_dates, value_date_index, correl_matrix,risk_free_rate, vols):
    '''
  Inputs:
  -------
  * S_0 (float): initial spot prices of the assets
  * num_asian_dates (int): number of asian dates. Notice that this includes the initial date, which fixes the initial spot values.
  * value_date_index (int): index, within the array of asian dates indicating the value date. (Actual step)
  * risk_free_rate (float): risk free rate
  * num_assets (int): number of underlying assets.
  * vols (array(float)): array of indiv assets vols.
  * correl_matrix (array(float, float)): matrix of correlations
  * initial_maturity (float): maturity of the product as seen on initial spot fixing date. (Years)
  * num_sims (int): number of simulations
  Outputs:
  --------
  * rets (numpy.ndarray): simulation of the evolution of the assets. Shape (num_sims, num_asian_dates-1, num_assets)
  * payoff (numpy.ndarray): Payoff of every simulation done. Shape (num_sims, )
  * premium (numpy.float64): present value in the actual step given by value_date_index.
  '''

    # Inputs:

    # Simulation of assets up to value date:

    init_time_array = np.linspace(0, initial_maturity, num_asian_dates)

    delta_t = initial_maturity / (num_asian_dates - 1)
    num_steps = value_date_index
    num_remaining_steps = num_asian_dates - value_date_index -1

    # Independent brownians
    inc_W = np.random.normal(size=(num_assets, num_steps), scale = np.sqrt(delta_t))

    # Cholesky matrix
    m = cholesky(correl_matrix)

    # Correlated brownians
    inc_W_correl = np.matmul(m, inc_W)

    # Independent brownian motion (the axis order is done to be able to correlate them with a matrix multiplication)
    inc_W_remaining = np.random.normal(size=(num_sims, num_remaining_steps, num_assets), scale = np.sqrt(delta_t))

    # We correlate them
    inc_W_correl_remaining = np.matmul(inc_W_remaining, m.T)

    # We transpose the 3D matrix of correlated B. motion (path, asset, time step)

    inc_W_correl_remaining = inc_W_correl_remaining.transpose([0,2,1])

    aux = np.repeat(inc_W_correl[None,...],num_sims,axis=0)

    # We attach the brownians obtained from t= 0 to value date

    inc_W_correl_total = np.concatenate((aux, inc_W_correl_remaining), axis = 2)

    # We compute exponential returns

    gross_rets_total = np.exp((risk_free_rate - 0.5 *vols **2) * delta_t + vols * inc_W_correl_total)

    # We simulate the underlyings

    S_T = np.cumprod(np.concatenate((np.repeat(S_0.reshape(-1,1)[None,...],num_sims,axis=0), gross_rets_total), axis = 2), axis = 2)

    # We compute the returns

    rets = S_T[:,:,1:] / np.repeat(S_0.reshape(-1,1)[None,...],num_sims,axis=0)

    payoff = np.maximum(np.sum(np.prod(rets, axis = (2))**(1/(num_assets * (num_asian_dates-1))), axis = 1)-3,0)
    print(payoff)

    # Calculate the premium as the discounted average value

    premium = np.mean(payoff*np.exp(-risk_free_rate*(initial_maturity-init_time_array[value_date_index])))

    rets = rets.transpose([0,2,1])

    return rets, payoff, premium

In [ ]:
num_steps = 20
vols = np.array([0.2,0.2,0.3]).reshape(-1,1)
risk_free_rate = 0.1


TTM = initial_maturity = 5 #years
value_date_index = 5 #actual step
num_asian_dates = num_steps + 1 #Time steps + 1
init_time_array = np.linspace(0, initial_maturity, num_asian_dates)

rho = 0.1
correl_matrix = rho = np.array([[1,0.8,0.2], [0.8,1,0.6],[0.2,0.6,1]])


num_sims = 100000

num_assets = 3 #numero de activos

rets, payoff, premium = MonteCarlo(initial_maturity, S_0, num_sims, num_assets, num_asian_dates, value_date_index, correl_matrix,risk_free_rate, vols)

[0.03072588 0.23885778 0.36191412 ... 0.28564543 0.66509811 0.35081853]


### Valor presente

Dados todos los timesteps:

In [18]:
#present_value (numpy.ndarray): present value of the payoff in every time step. Shape (num_sims, num_asian_dates-1)

time_array = np.linspace(0, initial_maturity, num_steps)

present_value = np.matmul(payoff.reshape(-1,1),np.exp(- risk_free_rate*(initial_maturity-time_array )).reshape(1,-1))

## Exportación de los datos

In [23]:
np.save(os.path.join('..', 'DATOS', 'TRANSFORMADOS', 'datos_sinteticos.npy'), rets)